# T029 — PREPARED_NOT_TRAINED
Bản chuẩn bị GenZ500, protocol synthetic_development/G2-S; T028 REAL vẫn BLOCKED. Train chỉ khi G2-S verified, CUDA và exact config/resource/cost receipt đã được duyệt. Không dùng synthetic làm blind evaluation. Notebook chưa được chạy trong phiên này. Không tự upload dữ liệu/checkpoint ra ngoài.

Current completion note: the user-authorized bounded local CPU classifier-head experiment executed separately via configs/t029_genz500_cpu_head.json (534steps, encoder frozen). This notebook remains the optional full-encoder GPU recipe, unexecuted and not approved for paid execution; it is not evidence of fullGPU training. See reports/ai_update/all_tasks_cpu_20261103.


In [ ]:
# Môi trường Colab đã được người dùng chọn; cài package nếu được phép.
%pip install transformers==4.57.3 huggingface-hub==0.36.2 tokenizers==0.22.2 safetensors==0.8.0

Sau cài đặt: restart runtime, rồi chỉ chạy các ô tiếp theo. Không Run all bỏ qua bước restart. Sao chép repo/checkpoint/dữ liệu theo phạm vi được người dùng cho phép; không upload tập test hoặc dữ liệu riêng chưa được duyệt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT=Path('/content/drive/MyDrive/ShieldVI_V3_Hybrid_Review')
BASE=Path('/content/drive/MyDrive/visobert_toxic_final')
assert BASE.is_dir()

CONFIG=PROJECT/"configs/t029_genz500_candidate.json"
PREPARED=Path("/content/t029_genz500_prepared") # scratch output, separate from Windows audit artifacts


In [ ]:
# Optional archive import only after the user permits moving the source bundle to Drive.
# Existing project folders are preserved; no old session is deleted.
import os,stat,subprocess,sys,zipfile
from pathlib import PurePosixPath
PROJECT_ZIP=None # optionally set an already-uploaded source ZIP path

def unpack_project(archive_path,target):
    target=target.resolve()
    if target.exists():raise RuntimeError('Project exists; use it or choose a new folder, never delete the old session')
    with zipfile.ZipFile(archive_path) as archive:
        if sum(item.file_size for item in archive.infolist())>2_000_000_000:raise ValueError('Source archive size budget exceeded')
        for item in archive.infolist():
            name=PurePosixPath(item.filename.replace(chr(92),'/'))
            final=(target/str(name)).resolve()
            if name.is_absolute() or '..' in name.parts or not final.is_relative_to(target):raise ValueError('ZIP path escapes extraction directory')
            if stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:raise ValueError('ZIP symlinks are not allowed')
        target.mkdir(parents=True)
        archive.extractall(target)

if PROJECT_ZIP is not None:unpack_project(PROJECT_ZIP,PROJECT)
AI_ROOT=PROJECT/'services/moderation-ai'
LEGACY_UI=PROJECT/'apps/moderation-web/legacy-v3'
assert AI_ROOT.is_dir() and LEGACY_UI.is_dir(), 'Use the complete canonical ShieldVI repository layout'
# Resolve the actual model adapter without starting a service, loading weights or opening a DB.
subprocess.run([sys.executable,'-c','from app.predictor import ViSoBERTPredictor; print(ViSoBERTPredictor.__module__)'],cwd=AI_ROOT,
    env={**os.environ,'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1'},check=True)
os.chdir(PROJECT)


In [ ]:
import torch,transformers,importlib.metadata
assert transformers.__version__=='4.57.3'
assert importlib.metadata.version('huggingface-hub')=='0.36.2'
assert torch.cuda.is_available(), 'GPU required; no CPU fallback'
print(torch.cuda.get_device_name(0))

In [ ]:
!python scripts/prepare_t029_training.py --config "$CONFIG" --base-model "$BASE" --output "$PREPARED"

In [ ]:
!python scripts/train_t029_candidate.py --config "$CONFIG" --base-model "$BASE" --prepared "$PREPARED"

Nếu preflight báo BLOCKED thì dừng. Receipt approval phải được ghi sau khi người dùng duyệt đúng SHA config và prepared manifest, resource=Colab_GPU, max_training_seconds=3600, cost_acknowledged=true. Không tự tạo receipt đã duyệt.

In [ ]:
# Chỉ bỏ comment sau khi preflight READY và receipt đã được duyệt thật.
# !python scripts/train_t029_candidate.py --config "$CONFIG" --base-model "$BASE" --prepared "$PREPARED" --approval /content/drive/MyDrive/T029_approval.json --run

Candidate lưu ở output_dir mới trong configs/t029_genz500_candidate.json. Báo cáo validation và reload smoke phải PASS; không tự deploy, không sửa threshold 0.41, không chạy test để chọn lại model. Đánh giá paired thuộc T030 khi đủ dependency.